<a href="https://colab.research.google.com/github/liangjingyi1211/neural-search-ranking/blob/main/notebooks/00_environment_setup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone https://github.com/liangjingyi1211/neural-search-ranking.git

Cloning into 'neural-search-ranking'...
remote: Enumerating objects: 4, done.
remote: Counting objects: 100% (4/4), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 4 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (4/4), done.


In [2]:
%cd /content/neural-search-ranking

/content/neural-search-ranking


In [3]:
!pwd
!ls -la

/content/neural-search-ranking
total 24
drwxr-xr-x 3 root root 4096 Nov  8 08:59 .
drwxr-xr-x 1 root root 4096 Nov  8 08:59 ..
drwxr-xr-x 8 root root 4096 Nov  8 08:59 .git
-rw-r--r-- 1 root root 4664 Nov  8 08:59 .gitignore
-rw-r--r-- 1 root root   83 Nov  8 08:59 README.md


In [4]:
from pathlib import Path

folders = [
    "configs",
    "data",
    "notebooks",
    "src/data",
    "src/retrieval",
    "src/training",
    "src/evaluation",
    "src/api",
    "tests",
    "reports",
    "app",
]

for folder in folders:
    Path(folder).mkdir(parents=True, exist_ok=True)
    Path(folder, ".gitkeep").touch(exist_ok=True)

print("Project directories created successfully!")

Project directories created successfully!


In [5]:
!find . -maxdepth 3 -type d

.
./app
./.git
./.git/refs
./.git/refs/remotes
./.git/refs/heads
./.git/refs/tags
./.git/branches
./.git/objects
./.git/objects/pack
./.git/objects/info
./.git/hooks
./.git/info
./.git/logs
./.git/logs/refs
./tests
./notebooks
./src
./src/retrieval
./src/evaluation
./src/training
./src/data
./src/api
./reports
./configs
./data


安装基础包

In [6]:
%pip install -q \
    sentence-transformers \
    faiss-cpu \
    beir \
    rank-bm25 \
    pandas \
    matplotlib \
    scikit-learn \
    pyyaml \
    pytest

验证依赖

In [7]:
import torch
import transformers
import sentence_transformers
import faiss
import beir
import pandas
import numpy
import rank_bm25

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Sentence Transformers:", sentence_transformers.__version__)
print("FAISS:", faiss.__version__)
print("NumPy:", numpy.__version__)

print("All core dependencies imported successfully!")

PyTorch: 2.14.1+cpu
Transformers: 5.22.1
Sentence Transformers: 5.7.0
FAISS: 1.15.1
NumPy: 2.4.6
All core dependencies imported successfully!


记录环境版本

In [8]:
import sys
import subprocess
from pathlib import Path

Path("reports").mkdir(exist_ok=True)

with open("reports/environment.txt", "w") as f:
    f.write(f"Python: {sys.version}\n")
    f.write(f"PyTorch: {torch.__version__}\n")
    f.write(f"CUDA: {torch.version.cuda}\n")
    f.write(
        subprocess.check_output(
            [sys.executable, "-m", "pip", "freeze"],
            text=True
        )
    )

print("Environment recorded.")

Environment recorded.


挂载Google drive

In [9]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


创建持久化实验目录

In [10]:
from pathlib import Path

DRIVE_ROOT = Path(
    "/content/drive/MyDrive/neural-search-ranking"
)

for folder in [
    "datasets",
    "checkpoints",
    "artifacts",
    "reports"
]:
    (DRIVE_ROOT / folder).mkdir(
        parents=True,
        exist_ok=True
    )

print("Persistent storage:", DRIVE_ROOT)

Persistent storage: /content/drive/MyDrive/neural-search-ranking


下载Scifact

In [11]:
from beir import util
from beir.datasets.data_loader import GenericDataLoader

dataset = "scifact"

url = (
    "https://public.ukp.informatik.tu-darmstadt.de/"
    f"thakur/BEIR/datasets/{dataset}.zip"
)

data_path = util.download_and_unzip(
    url,
    "data"
)

print("Dataset path:", data_path)

data/scifact.zip:   0%|          | 0.00/2.69M [00:00<?, ?iB/s]

Dataset path: data/scifact


查看数据

In [12]:
from beir import util
from beir.datasets.data_loader import GenericDataLoader

dataset = "scifact"

url = (
    "https://public.ukp.informatik.tu-darmstadt.de/"
    f"thakur/BEIR/datasets/{dataset}.zip"
)

data_path = util.download_and_unzip(
    url,
    "data"
)

print("Dataset path:", data_path)

Dataset path: data/scifact


使用预训练模型把文本编码为向量

In [13]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "BAAI/bge-small-en-v1.5",
    device="cuda" if torch.cuda.is_available() else "cpu"
)

sentences = [
    "How does the stock market work?",
    "Understanding stock prices and financial markets",
    "How to cook pasta at home"
]

embeddings = model.encode(
    sentences,
    normalize_embeddings=True
)

print("Embedding shape:", embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  133MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding shape: (3, 384)


计算相似度

In [14]:
import numpy as np

scores = embeddings[0] @ embeddings[1:].T

for text, score in zip(sentences[1:], scores):
    print(f"{score:.4f} | {text}")

0.7909 | Understanding stock prices and financial markets
0.4587 | How to cook pasta at home


In [15]:
!git config user.name "liangjingyi1211"
!git config user.email "1543995436@qq.com"

!git add .
!git commit -m "Initialize project structure"

[main 99f8de0] Initialize project structure
 17 files changed, 8205 insertions(+)
 create mode 100644 app/.gitkeep
 create mode 100644 configs/.gitkeep
 create mode 100644 data/.gitkeep
 create mode 100644 data/scifact.zip
 create mode 100644 data/scifact/corpus.jsonl
 create mode 100644 data/scifact/qrels/test.tsv
 create mode 100644 data/scifact/qrels/train.tsv
 create mode 100644 data/scifact/queries.jsonl
 create mode 100644 notebooks/.gitkeep
 create mode 100644 reports/.gitkeep
 create mode 100644 reports/environment.txt
 create mode 100644 src/api/.gitkeep
 create mode 100644 src/data/.gitkeep
 create mode 100644 src/evaluation/.gitkeep
 create mode 100644 src/retrieval/.gitkeep
 create mode 100644 src/training/.gitkeep
 create mode 100644 tests/.gitkeep
